<a href="https://colab.research.google.com/github/prince24-web/Mechine_learning/blob/main/RUG_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#mount from google drive
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [3]:
import pandas as pd

# Specify the path to your CSV file in Google Drive
# Replace 'My Drive/path/to/your/final_dataset.csv' with the actual path to your file
csv_file_path = '/content/drive/MyDrive/final_dataset.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(csv_file_path)

# Display the first 5 rows of the DataFrame
display(df.head())

,price_usd,fdv_usd,market_cap_usd,reserve_in_usd,volume_1h,volume_6h,volume_24h,price_change_1h,price_change_6h,price_change_24h,buys_1h,sells_1h,buys_24h,sells_24h,sell_ratio_24h,pool_age_days,label
0,0.000010,9.621265e+03,5923317.5,5.359747e-16,15.847057,15.847057,15.847057,0.000,0.000,0.000,1.0,0.0,1.0,0.0,0.0000,0,1
1,0.000012,1.167200e+06,5923317.5,1.426240e+01,503.640653,503.640653,503.640653,21.137,21.137,21.137,6.0,0.0,6.0,0.0,0.0000,0,1
2,0.000030,3.021339e+04,5923317.5,-2.736031e-02,0.105669,0.105669,0.105669,0.000,0.000,0.000,1.0,0.0,1.0,0.0,0.0000,0,1
3,0.000289,2.891230e+04,5923317.5,4.093247e+04,7077.513862,7077.513862,7077.513862,62.418,62.418,62.418,16.0,1.0,16.0,1.0,0.0588,0,1
4,0.000026,2.613493e+06,5923317.5,1.514585e+04,9155.398004,9155.398004,9155.398004,162.483,162.483,162.483,44.0,27.0,44.0,27.0,0.3803,0,1


In [4]:
#Feature Engineering

# Calculate fdv_to_reserve_ratio
df['fdv_to_reserve_ratio'] = df['fdv_usd'] / df['reserve_in_usd']

# Calculate mcap_to_reserve_ratio
df['mcap_to_reserve_ratio'] = df['market_cap_usd'] / df['reserve_in_usd']

# Calculate volume_concentration
df['volume_concentration'] = df['volume_1h'] / df['volume_24h']

# Calculate buy_sell_imbalance
df['buy_sell_imbalance'] = df['buys_24h'] - df['sells_24h']

# Calculate price_crash_score
df['price_crash_score'] = df['price_change_24h'] - df['price_change_1h']

# Display the DataFrame with new features
display(df.head())

,price_usd,fdv_usd,market_cap_usd,reserve_in_usd,volume_1h,volume_6h,volume_24h,price_change_1h,price_change_6h,price_change_24h,...,buys_24h,sells_24h,sell_ratio_24h,pool_age_days,label,fdv_to_reserve_ratio,mcap_to_reserve_ratio,volume_concentration,buy_sell_imbalance,price_crash_score
0,0.000010,9.621265e+03,5923317.5,5.359747e-16,15.847057,15.847057,15.847057,0.000,0.000,0.000,...,1.0,0.0,0.0000,0,1,1.795097e+19,1.105149e+22,1.0,1.0,0.0
1,0.000012,1.167200e+06,5923317.5,1.426240e+01,503.640653,503.640653,503.640653,21.137,21.137,21.137,...,6.0,0.0,0.0000,0,1,8.183757e+04,4.153100e+05,1.0,6.0,0.0
2,0.000030,3.021339e+04,5923317.5,-2.736031e-02,0.105669,0.105669,0.105669,0.000,0.000,0.000,...,1.0,0.0,0.0000,0,1,-1.104278e+06,-2.164931e+08,1.0,1.0,0.0
3,0.000289,2.891230e+04,5923317.5,4.093247e+04,7077.513862,7077.513862,7077.513862,62.418,62.418,62.418,...,16.0,1.0,0.0588,0,1,7.063414e-01,1.447095e+02,1.0,15.0,0.0
4,0.000026,2.613493e+06,5923317.5,1.514585e+04,9155.398004,9155.398004,9155.398004,162.483,162.483,162.483,...,44.0,27.0,0.3803,0,1,1.725551e+02,3.910852e+02,1.0,17.0,0.0


In [5]:
#drop some redundant features
# Drop only these — everything else stays
df.drop(columns=[
    'fdv_usd',
    'market_cap_usd',
    'volume_1h',
    'reserve_in_usd',
    'buys_1h',
    'sells_1h',
], inplace=True)

print(df.columns.tolist())
print(df.shape)

['price_usd', 'volume_6h', 'volume_24h', 'price_change_1h', 'price_change_6h', 'price_change_24h', 'buys_24h', 'sells_24h', 'sell_ratio_24h', 'pool_age_days', 'label', 'fdv_to_reserve_ratio', 'mcap_to_reserve_ratio', 'volume_concentration', 'buy_sell_imbalance', 'price_crash_score']
(2072, 16)


In [6]:
#split data for training
from sklearn.model_selection import train_test_split

X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,        # randomizes row order before splitting
    stratify=y,          # ensures both splits have same rug:legit ratio
    random_state=42      # reproducible results
)


In [7]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [8]:
#Random Forest model Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-Score: 1.0000


In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report

# ── 1. Check feature distributions by label ─────────────────────
print("=== Feature means by label ===")
print(df.groupby("label")[X.columns].mean().T.to_string())

# ── 2. Cross-validation (harder to cheat than a single split) ───
model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=5, scoring="f1")
print(f"\n=== 5-Fold CV F1 Scores ===")
print(cv_scores)
print(f"Mean F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ── 3. Feature importance ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, stratify=y, random_state=42
)
model.fit(X_train, y_train)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== Feature Importances ===")
print(importance.to_string(index=False))

=== Feature means by label ===
label                             0             1
price_usd              5.425867e+02  2.297153e+01
volume_6h              1.506019e+03  1.588299e+04
volume_24h             6.426939e+07  1.588299e+04
price_change_1h        6.247360e-02  4.258030e+04
price_change_6h        1.709499e-01  4.275622e+04
price_change_24h       1.850539e+00  4.275622e+04
buys_24h               2.143014e+00  5.638976e+01
sells_24h              1.127613e-01  3.077953e+01
sell_ratio_24h         7.794279e-04  2.537543e-01
pool_age_days          3.686793e+02  0.000000e+00
fdv_to_reserve_ratio   1.561605e+05  7.067311e+16
mcap_to_reserve_ratio  1.359937e+05  4.350980e+19
volume_concentration   2.327700e-02  6.119997e-01
buy_sell_imbalance     2.030253e+00  2.561024e+01
price_crash_score      1.788066e+00  1.759191e+02

=== 5-Fold CV F1 Scores ===
[0.41129032 0.99009901 1.         1.         1.        ]
Mean F1: 0.8803 ± 0.2345

=== Feature Importances ===
              feature  import

In [10]:
# Drop leaking and corrupted features
df.drop(columns=[
    'pool_age_days',          # leakage — rugs are age 0, legit are age 365+
    'fdv_to_reserve_ratio',   # corrupted — division by near-zero median
    'mcap_to_reserve_ratio',  # corrupted — same reason
    'price_change_1h',        # artifact — 42,000% values are not real
    'price_change_6h',        # artifact — same
    'price_change_24h',       # artifact — same
    'price_crash_score',      # derived from corrupted price change columns
], inplace=True)

# Cap extreme values in remaining features (winsorize at 99th percentile)
for col in ['volume_6h', 'volume_24h', 'volume_concentration',
            'buy_sell_imbalance', 'buys_24h', 'sells_24h']:
    cap = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap)

print("Clean features:", df.columns.tolist())
print(df.shape)

Clean features: ['price_usd', 'volume_6h', 'volume_24h', 'buys_24h', 'sells_24h', 'sell_ratio_24h', 'label', 'volume_concentration', 'buy_sell_imbalance']
(2072, 9)


In [11]:
#split and train
X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=True,        # randomizes row order before splitting
    stratify=y,          # ensures both splits have same rug:legit ratio
    random_state=42      # reproducible results
)


In [12]:
#Retrain Random Forest model
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [13]:
# Evaluate the new Random Forest model
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

new_y_pred = rf.predict(X_test)

new_accuracy = accuracy_score(y_test, new_y_pred)
new_precision = precision_score(y_test, new_y_pred)
new_recall = recall_score(y_test, new_y_pred)
new_f1 = f1_score(y_test, new_y_pred)

print(f"New Model Accuracy: {new_accuracy:.4f}")
print(f"New Model Precision: {new_precision:.4f}")
print(f"New Model Recall: {new_recall:.4f}")
print(f"New Model F1-Score: {new_f1:.4f}")

New Model Accuracy: 1.0000
New Model Precision: 1.0000
New Model Recall: 1.0000
New Model F1-Score: 1.0000


In [14]:
# ── 1. Check feature distributions by label ─────────────────────
print("=== Feature means by label ===")
print(df.groupby("label")[X.columns].mean().T.to_string())

# ── 2. Cross-validation (harder to cheat than a single split) ───
model = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=5, scoring="f1")
print(f"\n=== 5-Fold CV F1 Scores ===")
print(cv_scores)
print(f"Mean F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# ── 3. Feature importance ────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, stratify=y, random_state=42
)
model.fit(X_train, y_train)

importance = pd.DataFrame({
    "feature": X.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== Feature Importances ===")
print(importance.to_string(index=False))

=== Feature means by label ===
label                            0             1
price_usd             5.425867e+02     22.971531
volume_6h             1.517773e+02  11583.911520
volume_24h            5.356126e+06  15882.991336
buys_24h              2.142233e+00     45.226772
sells_24h             1.127613e-01     21.551850
sell_ratio_24h        7.794279e-04      0.253754
volume_concentration  2.327700e-02      0.612000
buy_sell_imbalance    1.995847e+00     20.614173

=== 5-Fold CV F1 Scores ===
[0.41129032 0.96969697 1.         1.         1.        ]
Mean F1: 0.8762 ± 0.2327

=== Feature Importances ===
             feature  importance
           volume_6h    0.302284
            buys_24h    0.165227
      sell_ratio_24h    0.150766
           sells_24h    0.136685
  buy_sell_imbalance    0.102057
          volume_24h    0.060423
           price_usd    0.044156
volume_concentration    0.038401
